[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/youtube.ipynb)

# YouTube Data API

Collect data from YouTube with the Data API v3 through the `google-api-python-client` SDK: search videos by keyword, look up a channel, fetch statistics for videos, and download the comments of a video. The four sections share one API client and are meant to be run in order.

**Setup.** Install `google-api-python-client` and `python-dotenv`. Create an
API key in the [Google Cloud console](https://console.cloud.google.com/apis/credentials)
with the YouTube Data API v3 enabled, and put it in a `.env` file next to the
notebook:

```
YOUTUBE_API_KEY=your-key
```

Never commit the `.env` file. In Google Colab there is no `.env` file, so set
the value with `os.environ["YOUTUBE_API_KEY"] = "..."` in a cell you delete
before sharing, or use Colab's Secrets panel.

Every request costs quota. The default is 100 `search().list` calls per day,
plus 10,000 units per day for every other read at 1 unit each. Every page
of results counts again. The quota resets at midnight Pacific time.
Reference: [YouTube Data API v3](https://developers.google.com/youtube/v3/docs).

## Build the client

In [1]:
import os 

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
youtube_api_key = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=youtube_api_key)

## Search videos

Search videos by keyword with `search().list`, filter by publication date, and read the first page of results. Searches have their own budget of 100 calls per day, so use them sparingly.

In [3]:
request = youtube.search().list(
        part="snippet",
        maxResults=25,
        q="cat",
        publishedAfter="2025-09-05T00:00:00Z"
    )
response = request.execute()

In [4]:
type(response)

dict

In [5]:
response.keys()

dict_keys(['kind', 'etag', 'nextPageToken', 'regionCode', 'pageInfo', 'items'])

In [6]:
response['nextPageToken']

'CBkQAA'

In [7]:
response['pageInfo']

{'totalResults': 843269, 'resultsPerPage': 25}

In [8]:
response['items'][0]

{'kind': 'youtube#searchResult',
 'etag': 'uB7igBGWEXBk1NY9T9ZrFUofBkQ',
 'id': {'kind': 'youtube#video', 'videoId': 'aM5Zlq0bPKY'},
 'snippet': {'publishedAt': '2025-09-06T08:19:24Z',
  'channelId': 'UC7wafFu5c8AO0YF5U7R7xFA',
  'title': '🔴 24/7 LIVE: Cat TV for Cats to Watch 😺 Cute Birds Squirrels Cat Games 4K',
  'description': 'Non-stop HDR live streaming for cats, dogs, parrots, or other nature lovers. Relaxing your pets can help minimize separation ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/aM5Zlq0bPKY/default_live.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/aM5Zlq0bPKY/mqdefault_live.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/aM5Zlq0bPKY/hqdefault_live.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Birder King',
  'liveBroadcastContent': 'live',
  'publishTime': '2025-09-06T08:19:24Z'}}

## Channel information

Look up a channel by handle with `channels().list` and read its statistics and metadata.

In [3]:
request = youtube.channels().list(
        part="contentDetails,id,localizations,snippet,statistics,status,topicDetails",
        forHandle="MissingSemester"
    )
response = request.execute()

In [4]:
response.keys()

dict_keys(['kind', 'etag', 'pageInfo', 'items'])

In [5]:
response['items'][0]

{'kind': 'youtube#channel',
 'etag': 'jh1zso5AMTFvHlLxB_6C90AjP3Y',
 'id': 'UCuXy5tCgEninup9cGplbiFw',
 'snippet': {'title': 'Missing Semester',
  'description': 'Classes teach you all about advanced topics within CS, from operating systems to machine learning, but there’s one critical subject that’s rarely covered, and is instead left to students to figure out on their own: proficiency with their tools. We’ll teach you how to master the command-line, use a powerful text editor, use fancy features of version control systems, and much more!\n\nStudents spend hundreds of hours using these tools over the course of their education (and thousands over their career), so it makes sense to make the experience as fluid and frictionless as possible. Mastering these tools not only enables you to spend less time on figuring out how to bend your tools to your will, but it also lets you solve problems that would previously seem impossibly complex.\n\nRead about the motivation behind this class at ht

## Video information

Fetch metadata and statistics for one or more videos by ID with `videos().list`. Up to 50 IDs fit in one call, separated by commas.

In [3]:
request = youtube.videos().list(
        part="snippet,statistics,contentDetails,status",
        id="Z56Jmr9Z34Q,kgII-YWo3Zw",
        maxResults=25,
    )
response = request.execute()

In [4]:
response.keys()

dict_keys(['kind', 'etag', 'items', 'pageInfo'])

In [5]:
response['pageInfo']

{'totalResults': 2, 'resultsPerPage': 2}

In [6]:
response['items']

[{'kind': 'youtube#video',
  'etag': '6OhYn0DVm5mVkSgrv1aqQ_Y-Z2M',
  'id': 'Z56Jmr9Z34Q',
  'snippet': {'publishedAt': '2020-02-02T03:40:08Z',
   'channelId': 'UCuXy5tCgEninup9cGplbiFw',
   'title': 'Lecture 1: Course Overview + The Shell (2020)',
   'description': 'You can find the lecture notes and exercises for this lecture at https://missing.csail.mit.edu/2020/course-shell/\n\nHelp us caption & translate this video!\n\nhttps://amara.org/v/C1Efe/',
   'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/Z56Jmr9Z34Q/default.jpg',
     'width': 120,
     'height': 90},
    'medium': {'url': 'https://i.ytimg.com/vi/Z56Jmr9Z34Q/mqdefault.jpg',
     'width': 320,
     'height': 180},
    'high': {'url': 'https://i.ytimg.com/vi/Z56Jmr9Z34Q/hqdefault.jpg',
     'width': 480,
     'height': 360},
    'standard': {'url': 'https://i.ytimg.com/vi/Z56Jmr9Z34Q/sddefault.jpg',
     'width': 640,
     'height': 480},
    'maxres': {'url': 'https://i.ytimg.com/vi/Z56Jmr9Z34Q/maxresdefault.jpg

## Video comments

Download the top-level comments of a video and their replies with `commentThreads().list`, one page at a time. Pass `nextPageToken` back as `pageToken` for the next page.

In [3]:
request = youtube.commentThreads().list(
        part="id,replies,snippet",
        videoId="Z56Jmr9Z34Q",
        maxResults=25
    )
response = request.execute()

In [4]:
response.keys()

dict_keys(['kind', 'etag', 'nextPageToken', 'pageInfo', 'items'])

In [5]:
response['nextPageToken']

'Z2V0X25ld2VzdF9maXJzdC0tQ2dnSWdBUVZGN2ZST0JJRkNJa2dHQUFTQlFpSElCZ0FFZ1VJblNBWUFSSUZDSWdnR0FBU0JRaW9JQmdBSWc0S0RBam82OEdzQmhESWdMSzNBdw=='

In [6]:
response['pageInfo']

{'totalResults': 25, 'resultsPerPage': 25}

In [7]:
response['items']

[{'kind': 'youtube#commentThread',
  'etag': '2AOlty1IV79_wpd5i8Dz42YKk3A',
  'id': 'UgxLg0cEkOPoQmrvkn94AaABAg',
  'snippet': {'channelId': 'UCuXy5tCgEninup9cGplbiFw',
   'videoId': 'Z56Jmr9Z34Q',
   'topLevelComment': {'kind': 'youtube#comment',
    'etag': 'LFxv-PsWxpD0F8Ddpt9W2mHXvc8',
    'id': 'UgxLg0cEkOPoQmrvkn94AaABAg',
    'snippet': {'channelId': 'UCuXy5tCgEninup9cGplbiFw',
     'videoId': 'Z56Jmr9Z34Q',
     'textDisplay': 'maybe this will work: sudo find / -name &#39;*brightness*&#39;',
     'textOriginal': "maybe this will work: sudo find / -name '*brightness*'",
     'authorDisplayName': '@anfieldxu',
     'authorProfileImageUrl': 'https://yt3.ggpht.com/ytc/AIdro_lY7YOo1-A7lJk4VDxh5jNEkmGDnuLqeC0qDSp7bVcpfSk=s48-c-k-c0x00ffffff-no-rj',
     'authorChannelUrl': 'http://www.youtube.com/@anfieldxu',
     'authorChannelId': {'value': 'UCwSvG_r7IJFnDLdLvBgbABg'},
     'canRate': True,
     'viewerRating': 'none',
     'likeCount': 0,
     'publishedAt': '2025-06-23T20:24:10Z'